In [105]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
from IPython.display import display, Markdown
import os
load_dotenv(override=True)

True

In [65]:
checklist = []
completed = []

### display of text

In [68]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [69]:
show("here is [yellow]a[/yellow] rich [strike]console[/strike] print")

here is a rich console print

### adds the local openAI

In [70]:
lmstudio = OpenAI(base_url='http://localhost:1234/v1/', api_key='lmstudio')
model_name = "unsloth-gemma-4-26b-a4b-it-qat-oq4"

### checklist functions

1. get_checklist_report – displays the list, with strikeouts when done. 
2. create_check_list – when passed a list, creates a list
3. mark_complete - passed an index, and notes on the completion. 

2 and 3 are both tools to be passed to the llm, so need JSON descriptions to go with them 

In [80]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [81]:
get_checklist_report()

''

In [82]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [83]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [84]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [85]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

### local tool handling 

1. create a json of the tools for the llm. To be passed to the llm chat function
2. handles tools

In [86]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]

In [87]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [88]:
def loop(messages):
    response = lmstudio.chat.completions.create(model="gpt-5.5", messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = lmstudio.chat.completions.create(model="gpt-5.5", messages=messages, tools=tools)
    show(response.choices[0].message.content)

### prompt set up

In [106]:
weekTranscription = os.getenv('TRANSCRIPTION')

with open(weekTranscription, "r", encoding="utf-8") as f:
    summary = f.read()
display(Markdown(summary[:500]))

# Week 3: Repetition
*Module 2 – Chaos and Control*

## Contents

1. 1 hand coding repeats
2. 2 for loops
3. 3 loop DANGER
4. 4 nested loops
5. 5 maps

---

---
video_id: 1-hand-coding-repeats
video_title: 1 hand coding repeats
video_file: 1 hand coding repeats.mp4
video_url: "https://aacontent.b-cdn.net/classes/creativeCode/week%203/1%20hand%20coding%20repeats.mp4"
week: week 3
topic: The inefficiency of manual code duplication for creating repetitive patterns and the motivation for using loops

In [118]:
system_message = f"""
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
Here's a transcript of a lecture on the topic of the problem: \n{summary}\n
"""
user_message = """

Please generate 5 multiple-choice question concepts. All the questions should be relevant to the content of the lecture transcript, and be used to reinforce the key ideas.
For each question concept, generate 4 variations of the questions. Each question should have 4 answer options, and indicate which option is correct. Each of the variations should use a different answer option as the correct answer.
Each question should have one correct answer and three distractors.
The questions should be designed to test understanding of the key concepts and details presented in the lecture. 
One of the 5 questions, should be a question that requires the student to read some incomplete code, and choose the correct code to put in the gap.
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [119]:
def loop(messages):
    response = lmstudio.chat.completions.create(model="gpt-5.5", messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = lmstudio.chat.completions.create(model="gpt-5.5", messages=messages, tools=tools)
    show(response.choices[0].message.content)

In [120]:
checklist, completed = [], []
loop(messages)

Checklist #1: Concept 1: Anatomy and purpose of a for loop.
Checklist #2: Concept 2: Identifying infinite loops (Convergence/Direction).
Checklist #3: Concept 3: Understanding Nested Loops execution order.
Checklist #4: Concept 4: The map() function parameters and purpose.
Checklist #5: Concept 5: Code completion (filling the gap in a loop).

Generated 4 variations for Concept 1.
Variation 1 (A): What are the three components of a for loop?
Variation 2 (B): Which part of the for loop determines if the loop continues or stops?
Variation 3 (C): What is the purpose of the 'initialization' in a for loop?
Variation 4 (D): Which part of the for loop updates the counter variable after each iteration?

Checklist #1: Concept 1: Anatomy and purpose of a for loop.
Checklist #2: Concept 2: Identifying infinite loops (Convergence/Direction).
Checklist #3: Concept 3: Understanding Nested Loops execution order.
Checklist #4: Concept 4: The map() function parameters and purpose.
Checklist #5: Concept 5: Code completion (filling the gap in a loop).

Generated 4 variations for Concept 2.
Variation 1 (A): Which scenario causes an infinite loop?
Variation 2 (B): What is the 'Golden Rule' for preventing infinite loops?
Variation 3 (C): Why does `for (let x = 0; x < 10; x--)` cause a crash?
Variation 4 (D): What happens if the update step moves the variable away from the condition?

Checklist #1: Concept 1: Anatomy and purpose of a for loop.
Checklist #2: Concept 2: Identifying infinite loops (Convergence/Direction).
Checklist #3: Concept 3: Understanding Nested Loops execution order.
Checklist #4: Concept 4: The map() function parameters and purpose.
Checklist #5: Concept 5: Code completion (filling the gap in a loop).

...

Generated 4 variations for Concept 4.
Variation 1 (A): What is the primary purpose of the map() function?
Variation 2 (B): In `map(value, start1, stop1, start2, stop2)`, what do `start1` and `stop1` represent?
Variation 3 (C): How can you use the map() function to create an inverted gradient?
Variation 4 (D): What is a common mistake when using the map() function in drawing code?

Checklist #1: Concept 1: Anatomy and purpose of a for loop.
Checklist #2: Concept 2: Identifying infinite loops (Convergence/Direction).
Checklist #3: Concept 3: Understanding Nested Loops execution order.
Checklist #4: Concept 4: The map() function parameters and purpose.
Checklist #5: Concept 5: Code completion (filling the gap in a loop).

Generated 4 variations for Concept 5.
Variation 1 (A): Complete the loop to draw 5 circles: `for (int x = 0; _____; x += 1) { circle(x, 50); }`
Variation 2 (B): Complete the loop to draw a vertical line: `for (int y = 0; _____; y += 10) { circle(50, y); }`
Variation 3 (C): Complete the loop to draw a grid: `for (int x = 0; x < 400; x += 50) { _____; }`
Variation 4 (D): Complete the loop to draw a row: `for (int i = 0; _____; i++) { rect(i, 10, 5, 5); }`

Checklist #1: Concept 1: Anatomy and purpose of a for loop.
Checklist #2: Concept 2: Identifying infinite loops (Convergence/Direction).
Checklist #3: Concept 3: Understanding Nested Loops execution order.
Checklist #4: Concept 4: The map() function parameters and purpose.
Checklist #5: Concept 5: Code completion (filling the gap in a loop).

<|channel>thought
<channel|># Multiple Choice Question Concepts

## Concept 1: Anatomy and purpose of a for loop

**Variation 1**
What are the three components required within the parentheses of a `for` loop?
A) Initialization, Condition, and Incrementer
B) Start value, End value, and Step size
C) Variable declaration, Boolean expression, and Update rule
D) Input value, Comparison operator, and Incrementer
**Correct Answer: A**

**Variation 2**
Which part of the `for` loop determines whether the loop continues to run or terminates?
A) The initialization statement
B) The condition (boolean expression)
C) The incrementer/update step
D) The code block inside the curly braces
**Correct Answer: B**

**Variation 3**
What is the purpose of the 'initialization' step in a `for` loop?
A) To define how many times the loop will run total.
B) To check if the loop should start or stop.
C) To declare and set the starting value of the loop counter variable.
D) To update the variable after each iteration is complete.
**Correct Answer: C**

**Variation 4**
Which part of the `for` loop is responsible for updating the counter variable after each iteration?
A) The initialization
B) The condition
C) The code block
D) The incrementer
**Correct Answer: D**

***

## Concept 2: Identifying infinite loops (Convergence/Direction)

**Variation 1**
Which of the following scenarios is most likely to cause an infinite loop?
A) A loop where the variable moves toward the exit condition.
B) A loop where the update step causes the variable to move away from the exit condition.
C) A loop where the condition is `x < 10` and the incrementer is `x++`.
D) A loop where the variable starts at 0 and ends at 10.
**Correct Answer: B**

**Variation 2**
What is the "Golden Rule" for preventing infinite loops in a `for` loop?
A) Always use the `<` operator instead of `<=`.
B) Ensure the variable undergoes convergence toward the exit condition.
C) Always initialize your loop counter at zero.
D) Never use a variable inside the condition that is also used in the incrementer.
**Correct Answer: B**

**Variation 3**
Why does `for (let x = 0; x < 10; x--)` cause a browser to crash?
A) Because `x` will always be less than 10 as it decreases, creating an infinite loop.
B) Because the incrementer is using a subtraction instead of addition.
C) Because `x` starts at 0, which is the wrong starting point for a decrementing loop.
D) Because the condition `x < 10` is always true for negative numbers.
**Correct Answer: A**

**Variation 4**
What happens if the update step in a loop moves the variable in the opposite direction of the exit condition?
A) The loop will run exactly one time and then stop.
B) The loop will trigger an "off-by-one" error.
C) The computer will run out of memory because the loop becomes infinite.
D) The code will automatically correct itself by reversing the update step.
**Correct Answer: C**

***

## Concept 3: Understanding Nested Loops execution order

**Variation 1**
In a nested loop structure, which loop completes its entire cycle first?
A) The inner loop
B) The outer loop
C) Both loops simultaneously
D) Neither; they run in parallel
**Correct Answer: A**

**Variation 2**
What is the correct order of execution for nested loops?
A) The outer loop runs once, then the inner loop runs to completion, then the outer loop moves to its next step.
B) The inner loop completes one full cycle for every single iteration of the outer loop.
C) Both loops iterate through their entire ranges before any drawing code is executed.
D) The outer loop and inner loop alternate steps one by one.
**Correct Answer: B**

**Variation 3**
If an outer loop runs 5 times and the inner loop runs 10 times, how many total iterations of the innermost code 
block occur?
A) 15
B) 50
C) 10
D) 5
**Correct Answer: B**

**Variation 4**
Which loop is responsible for moving the pattern to the next "column" or primary axis after a full row/column is 
finished?
A) The inner loop
B) The incrementer of the inner